In [1]:
import pandas as pd
import re
import unicodedata
from itables import show

# Ingesta de datos

In [2]:
# Ponemos la ruta donde se encuentra cada uno de los archivos
# el segundo arhcivo de cada uno de los DF refiere a la actualización de fechas

df_costeos_serv_raw_1 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/response.csv')
df_costeos_serv_raw_2 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/response2.csv')
df_costeos_serv_raw = pd.concat([df_costeos_serv_raw_1, df_costeos_serv_raw_2], ignore_index=True)

# mapa de servicios id
df_servicios_id = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/Todos Procesos Producción.csv')

# Costeos x PTO
df_costeos_pto1 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/Todos Costeo Producto.csv')
df_costeos_pto2 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/Todos Costeo Producto (1).csv')
df_costeos_pto = pd.concat([df_costeos_pto1, df_costeos_pto2], ignore_index=True)


# OP detalle
df_op_det1 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/File.csv')
df_op_det2 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/File (1).csv')
df_op_det = pd.concat([df_op_det1, df_op_det2], ignore_index=True)

# Ordenes de Servicio
df_os1 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/Todas OS.csv')
df_os2 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/Todas OS (1).csv')
df_os = pd.concat([df_os1, df_os2], ignore_index=True)


#Comercial
df_comercial1 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/Todos Costeos.csv')
df_comercial2 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/Todos Costeos (1).csv')
df_comercial = pd.concat([df_comercial1, df_comercial2], ignore_index=True)

# OP (Solo para tomar el valor total de la OP)
df_op1 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/Todas OP.csv')
df_op2 = pd.read_csv('C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/Todas OP_1.csv')
df_op = pd.concat([df_op1, df_op2], ignore_index=True)

### Actualización de fechas

In [3]:
df_costeos_serv_raw = df_costeos_serv_raw.drop('Unnamed: 0', axis=1)

In [4]:
df_costeos_serv_raw.columns = ['id_costeoxpto', 'id_serv', 'costo_antes_iva', 'fecha_creacion']

In [5]:
df_costeos_serv_raw.shape

(339276, 4)

In [6]:
df_costeos_serv_raw.describe()

,id_costeoxpto,id_serv,costo_antes_iva
count,3.391440e+05,3.392760e+05,3.392710e+05
mean,4.305538e+07,1.693140e+07,9.991461e+03
std,1.707540e+07,1.520339e+06,2.232408e+05
min,1.511720e+07,1.415802e+07,-2.000000e+03
25%,2.670623e+07,1.653823e+07,1.400000e+03
50%,4.172613e+07,1.654400e+07,3.000000e+03
75%,5.502015e+07,1.842954e+07,9.000000e+03
max,8.376411e+07,4.537002e+07,4.096208e+07


In [7]:
df_costeos_serv_raw = df_costeos_serv_raw.dropna(axis=0, subset=['id_costeoxpto'])
df_costeos_serv_raw = df_costeos_serv_raw.dropna(axis=0, subset=['costo_antes_iva'])

In [8]:
df_costeos_serv_raw = df_costeos_serv_raw.drop_duplicates()

In [9]:
df_costeos_serv_raw.shape

(123831, 4)

# Data Cleaning

In [10]:
# Homologación servicios costeos
df_costeos_serv = df_costeos_serv_raw.merge(
    df_servicios_id[["Id.", "Proceso Producción"]].rename(columns={"Id.": "id_serv"}),
    on="id_serv",
    how="left",
).drop(columns=["id_serv"])

# union con costeos x pto
df_costeos_pto = df_costeos_pto[
    ["Costeo_Producto", "Producto", "Cliente", "Cantidad Solicitada", "Id.", "Fecha"]
].rename(columns={"Producto":"Productos"})

df_costeos_pto = df_costeos_pto.merge(
    df_costeos_serv.rename(columns={"id_costeoxpto": "Id."}),
    on="Id.",
    how="left",
)

df_costeos_pto

,Costeo_Producto,Productos,Cliente,Cantidad Solicitada,Id.,Fecha,costo_antes_iva,fecha_creacion,Proceso Producción
0,COS-0012287-2,Chaqueta Térmica,Coorserpark,34,77898260,30/12/2025,1800.0,2025-12-30T12:41:33Z,Bordado
1,COS-0012287-2,Chaqueta Térmica,Coorserpark,34,77898260,30/12/2025,18000.0,2025-12-30T12:41:33Z,Confección
2,COS-0012288-1,CHALECO REFLECTIVO,TALMA COLOMBIA,2,77917786,30/12/2025,1500.0,2025-12-31T00:42:03Z,Estampado
3,COS-0012288-1,CHALECO REFLECTIVO,TALMA COLOMBIA,2,77917786,30/12/2025,12000.0,2025-12-31T00:39:59Z,Confección
4,COS-0012287-2,Gorra Curva,Coorserpark,4,77898321,30/12/2025,1800.0,2025-12-30T12:43:54Z,Bordado
...,...,...,...,...,...,...,...,...,...
100639,COS-0012459-6,Saco Tejido,AEROSAN S.A.S.,56,79429219,02/02/2026,1800.0,2026-02-02T06:04:38Z,Bordado
100640,COS-0012459-6,Saco Tejido,AEROSAN S.A.S.,56,79429219,02/02/2026,39000.0,2026-02-02T06:04:38Z,Paquete Completo Sin Marca
100641,COS-0012213-6,Saco Tejido,AEROSAN S.A.S.,30,79429249,02/02/2026,800.0,2026-02-02T06:11:59Z,Transporte
100642,COS-0012213-6,Saco Tejido,AEROSAN S.A.S.,30,79429249,02/02/2026,1800.0,2026-02-02T06:11:59Z,Bordado


In [11]:
df_costeos_pto.drop_duplicates(inplace=True)
df_costeos_pto.duplicated().sum()

np.int64(0)

In [12]:
df_op_det.drop_duplicates(inplace=True)
df_op_det.duplicated().sum()

np.int64(0)

In [13]:
# unir a OP detalle
df_op_det.drop(
    columns=[
        "ClienteNombre",
        "Categoría Proc",
        "Detalle",
        "Colores",
        "Total Pieza",
        "Cantidad Remitida",
        "Precio Unitario Sin IVA",
        "Estado OP",
        "Fecha Promesa",
        "Ficha Técnica Producto",
        "Muestra",
        "Id.",
        "OS",
        "Creado El"
    ],
    inplace=True,
)
df_op_det = df_op_det.rename(columns={'Id Costeo Producto' : 'Id.'})

In [14]:
df_op_det

,Num-OP,Secuencia,OP Det,Productos,Producir,Total Precio Sin IVA,Id.,Costeo_Producto
0,6672.0,1.0,6672-1-Buzo / Cerrado,Buzo / Cerrado,26,1559168.0,81884540.0,COS-0012801-2
1,6672.0,2.0,6672-2-Buzo / Cerrado,Buzo / Cerrado,26,1559168.0,81884548.0,COS-0012801-3
2,6672.0,3.0,6672-3-Buzo / Cerrado,Buzo / Cerrado,36,2158848.0,81884555.0,COS-0012801-4
3,6672.0,4.0,6672-4-Buzo / Cerrado,Buzo / Cerrado,36,2158848.0,81884562.0,COS-0012801-5
4,6671.0,1.0,6671-1-Abrigo Sastre,Abrigo Sastre,1,522104.0,81147999.0,COS-0012713-1
...,...,...,...,...,...,...,...,...
10483,6651.0,18.0,6651-18-Jogger,Jogger Jogger,1,42494.0,80783310.0,COS-0012673-76
10491,6649.0,1.0,6649-1-T-SHIRT MANGA CORTA,T-SHIRT MANGA CORTA,2,68994.0,81425526.0,COS-0012751-2
10492,6649.0,2.0,6649-2-Camibuzo,Camibuzo,2,82002.0,81425540.0,COS-0012751-4
10493,6649.0,3.0,6649-3-Pantalón dotación,Pantalón dotación,2,96974.0,81425453.0,COS-0012750-2


In [15]:
df_op_det = df_op_det.drop_duplicates()
df_op_det.duplicated().sum()

np.int64(0)

In [16]:
df_op_det = df_op_det.merge(
    df_costeos_pto,
    on=["Costeo_Producto", "Productos", "Id."],
    how="left",
)
df_op_det

,Num-OP,Secuencia,OP Det,Productos,Producir,Total Precio Sin IVA,Id.,Costeo_Producto,Cliente,Cantidad Solicitada,Fecha,costo_antes_iva,fecha_creacion,Proceso Producción
0,6672.0,1.0,6672-1-Buzo / Cerrado,Buzo / Cerrado,26,1559168.0,81884540.0,COS-0012801-2,GRAN AMERICA,1.0,17/03/2026,7000.0,2026-03-17T22:56:25Z,Confección
1,6672.0,1.0,6672-1-Buzo / Cerrado,Buzo / Cerrado,26,1559168.0,81884540.0,COS-0012801-2,GRAN AMERICA,1.0,17/03/2026,1500.0,2026-03-17T22:56:25Z,Estampado
2,6672.0,2.0,6672-2-Buzo / Cerrado,Buzo / Cerrado,26,1559168.0,81884548.0,COS-0012801-3,GRAN AMERICA,1.0,17/03/2026,7000.0,2026-03-17T22:56:25Z,Confección
3,6672.0,2.0,6672-2-Buzo / Cerrado,Buzo / Cerrado,26,1559168.0,81884548.0,COS-0012801-3,GRAN AMERICA,1.0,17/03/2026,1500.0,2026-03-17T22:56:25Z,Estampado
4,6672.0,3.0,6672-3-Buzo / Cerrado,Buzo / Cerrado,36,2158848.0,81884555.0,COS-0012801-4,GRAN AMERICA,1.0,17/03/2026,7000.0,2026-03-17T22:56:25Z,Confección
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19508,6674.0,4.0,6674-4-CHAQUETA SPORT,CHAQUETA SPORT,2,202146.0,81945656.0,COS-0012803-6,MAXIALIMENTOS,2.0,18/03/2026,7500.0,2026-03-18T20:09:16Z,Estampado
19509,6673.0,1.0,6673-1-POLO MANGA CORTA,POLO MANGA CORTA,1,23019.0,81946992.0,COS-0012804-2,NUTRESA,1.0,18/03/2026,5000.0,2026-03-18T20:51:57Z,Confección
19510,6673.0,1.0,6673-1-POLO MANGA CORTA,POLO MANGA CORTA,1,23019.0,81946992.0,COS-0012804-2,NUTRESA,1.0,18/03/2026,3000.0,2026-03-18T20:51:57Z,Bordado
19511,6673.0,2.0,6673-2-POLO MANGA CORTA,POLO MANGA CORTA,1,23019.0,81946893.0,COS-0012804-2,NUTRESA,1.0,18/03/2026,5000.0,2026-03-18T20:45:07Z,Confección


In [17]:
df_op_det = df_op_det.dropna(axis=0, subset=['costo_antes_iva'])

In [18]:
df_op_det

,Num-OP,Secuencia,OP Det,Productos,Producir,Total Precio Sin IVA,Id.,Costeo_Producto,Cliente,Cantidad Solicitada,Fecha,costo_antes_iva,fecha_creacion,Proceso Producción
0,6672.0,1.0,6672-1-Buzo / Cerrado,Buzo / Cerrado,26,1559168.0,81884540.0,COS-0012801-2,GRAN AMERICA,1.0,17/03/2026,7000.0,2026-03-17T22:56:25Z,Confección
1,6672.0,1.0,6672-1-Buzo / Cerrado,Buzo / Cerrado,26,1559168.0,81884540.0,COS-0012801-2,GRAN AMERICA,1.0,17/03/2026,1500.0,2026-03-17T22:56:25Z,Estampado
2,6672.0,2.0,6672-2-Buzo / Cerrado,Buzo / Cerrado,26,1559168.0,81884548.0,COS-0012801-3,GRAN AMERICA,1.0,17/03/2026,7000.0,2026-03-17T22:56:25Z,Confección
3,6672.0,2.0,6672-2-Buzo / Cerrado,Buzo / Cerrado,26,1559168.0,81884548.0,COS-0012801-3,GRAN AMERICA,1.0,17/03/2026,1500.0,2026-03-17T22:56:25Z,Estampado
4,6672.0,3.0,6672-3-Buzo / Cerrado,Buzo / Cerrado,36,2158848.0,81884555.0,COS-0012801-4,GRAN AMERICA,1.0,17/03/2026,7000.0,2026-03-17T22:56:25Z,Confección
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19508,6674.0,4.0,6674-4-CHAQUETA SPORT,CHAQUETA SPORT,2,202146.0,81945656.0,COS-0012803-6,MAXIALIMENTOS,2.0,18/03/2026,7500.0,2026-03-18T20:09:16Z,Estampado
19509,6673.0,1.0,6673-1-POLO MANGA CORTA,POLO MANGA CORTA,1,23019.0,81946992.0,COS-0012804-2,NUTRESA,1.0,18/03/2026,5000.0,2026-03-18T20:51:57Z,Confección
19510,6673.0,1.0,6673-1-POLO MANGA CORTA,POLO MANGA CORTA,1,23019.0,81946992.0,COS-0012804-2,NUTRESA,1.0,18/03/2026,3000.0,2026-03-18T20:51:57Z,Bordado
19511,6673.0,2.0,6673-2-POLO MANGA CORTA,POLO MANGA CORTA,1,23019.0,81946893.0,COS-0012804-2,NUTRESA,1.0,18/03/2026,5000.0,2026-03-18T20:45:07Z,Confección


In [19]:
# Limpieza de comercial

df_comercial.columns

# Eliminación de columnas no relevantes
df_comercial.drop(
    columns=[
        "Nombre",
        "Estado",
        "Margen Ponderado",
        "Creado en",
        "Acciones",
    ],
    inplace=True,
)
df_comercial.head()


,Costeo,Cliente,Nombre del Comercial,Id.
0,COS-0012876,AEROSAN S.A.S.,Pilar Gómez,82524713
1,COS-0012875,NUTRESA,Luis Guzman,82520897
2,COS-0012874,NUTRESA,Luis Guzman,82496746
3,COS-0012873,NUTRESA,Luis Guzman,82495472
4,COS-0012872,NUTRESA,Luis Guzman,82492851


In [20]:
df_op_det['Num-OP'] = pd.to_numeric(df_op_det['Num-OP'], errors='coerce').fillna(0).astype(int)
df_op_det['Secuencia'] = pd.to_numeric(df_op_det['Secuencia'], errors='coerce').fillna(0).astype(int)

df_op_det['OP-D'] = df_op_det['Num-OP'].astype(str) + '-' + df_op_det['Secuencia'].astype(str)

df_op_det['secuencia'] = df_op_det['Secuencia']

C:\Users\Laura\AppData\Local\Temp\ipykernel_18816\2285874593.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_op_det['Num-OP'] = pd.to_numeric(df_op_det['Num-OP'], errors='coerce').fillna(0).astype(int)
C:\Users\Laura\AppData\Local\Temp\ipykernel_18816\2285874593.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_op_det['Secuencia'] = pd.to_numeric(df_op_det['Secuencia'], errors='coerce').fillna(0).astype(int)
C:\Users\Laura\AppData\Local\Temp\ipykernel_18816\2285874593.py:4: SettingWithCopyWarnin

In [21]:
df_op_det

,Num-OP,Secuencia,OP Det,Productos,Producir,Total Precio Sin IVA,Id.,Costeo_Producto,Cliente,Cantidad Solicitada,Fecha,costo_antes_iva,fecha_creacion,Proceso Producción,OP-D,secuencia
0,6672,1,6672-1-Buzo / Cerrado,Buzo / Cerrado,26,1559168.0,81884540.0,COS-0012801-2,GRAN AMERICA,1.0,17/03/2026,7000.0,2026-03-17T22:56:25Z,Confección,6672-1,1
1,6672,1,6672-1-Buzo / Cerrado,Buzo / Cerrado,26,1559168.0,81884540.0,COS-0012801-2,GRAN AMERICA,1.0,17/03/2026,1500.0,2026-03-17T22:56:25Z,Estampado,6672-1,1
2,6672,2,6672-2-Buzo / Cerrado,Buzo / Cerrado,26,1559168.0,81884548.0,COS-0012801-3,GRAN AMERICA,1.0,17/03/2026,7000.0,2026-03-17T22:56:25Z,Confección,6672-2,2
3,6672,2,6672-2-Buzo / Cerrado,Buzo / Cerrado,26,1559168.0,81884548.0,COS-0012801-3,GRAN AMERICA,1.0,17/03/2026,1500.0,2026-03-17T22:56:25Z,Estampado,6672-2,2
4,6672,3,6672-3-Buzo / Cerrado,Buzo / Cerrado,36,2158848.0,81884555.0,COS-0012801-4,GRAN AMERICA,1.0,17/03/2026,7000.0,2026-03-17T22:56:25Z,Confección,6672-3,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19508,6674,4,6674-4-CHAQUETA SPORT,CHAQUETA SPORT,2,202146.0,81945656.0,COS-0012803-6,MAXIALIMENTOS,2.0,18/03/2026,7500.0,2026-03-18T20:09:16Z,Estampado,6674-4,4
19509,6673,1,6673-1-POLO MANGA CORTA,POLO MANGA CORTA,1,23019.0,81946992.0,COS-0012804-2,NUTRESA,1.0,18/03/2026,5000.0,2026-03-18T20:51:57Z,Confección,6673-1,1
19510,6673,1,6673-1-POLO MANGA CORTA,POLO MANGA CORTA,1,23019.0,81946992.0,COS-0012804-2,NUTRESA,1.0,18/03/2026,3000.0,2026-03-18T20:51:57Z,Bordado,6673-1,1
19511,6673,2,6673-2-POLO MANGA CORTA,POLO MANGA CORTA,1,23019.0,81946893.0,COS-0012804-2,NUTRESA,1.0,18/03/2026,5000.0,2026-03-18T20:45:07Z,Confección,6673-2,2


In [22]:
df_costeos_serv_raw.columns

Index(['id_costeoxpto', 'id_serv', 'costo_antes_iva', 'fecha_creacion'], dtype='object')

In [23]:
df_os.duplicated().sum()

np.int64(0)

In [24]:
# añadir prefijo OS a columnas de df_os
for col in df_os.columns:
    if col not in ['Num OS', 'Orden de Satélite', 'Satélite Procesos']:
        df_os.rename(columns={col: 'OS-' + col}, inplace=True)

df_os['OP-D'] = df_os['Orden de Satélite'].str.split('-').str[2] + '-' + df_os['Orden de Satélite'].str.split('-').str[3]

#Ponemos el "NUM OS" dentro del groupby para que no se pierna la información
df_os_grouped = df_os.groupby(['OP-D', 'Satélite Procesos', 'OS-Proveedor', 'Num OS']).agg({
    'OS-Valor Unitario': 'mean',
    'OS-Cantidad': 'sum'
}).reset_index()

In [25]:
df_os_grouped.duplicated().sum()

np.int64(0)

In [26]:
df_op_det.shape

(17498, 16)

In [27]:
# Costeo servicio op vs os
df_op_det_vs_os = df_op_det.merge(
    df_os_grouped.rename(columns={"Satélite Procesos": "Proceso Producción"}),
    on=["OP-D", "Proceso Producción"],
    how="left",
)


In [28]:
# Filtramos para quedarnos solo con las filas que no esten vacias en OS-Valor Unitario 
df_op_det_vs_os = df_op_det_vs_os[df_op_det_vs_os['OS-Valor Unitario'].notna()]

# Separamos el último número del costeo para poder unir con el df de comercial
df_op_det_vs_os['Costeo'] = (df_op_det_vs_os['Costeo_Producto'].str.rsplit('-', n=1).str[0])


In [29]:
df_op_det_vs_os.duplicated().sum()

np.int64(0)

In [30]:
df_comercial.duplicated().sum()

np.int64(165)

In [31]:
df_final = df_op_det_vs_os.merge(
    df_comercial,
    on=["Costeo", "Cliente"],
    how="left"
)

In [32]:
df_final = df_final[[
    'Num-OP', 'Secuencia', 'OP-D', 'OP Det', 'Productos', 'Cliente', 
    'Fecha', 'Id._x', 'Costeo_Producto', 'Costeo', 'Proceso Producción', 
    'Producir', 'OS-Cantidad', 'Num OS',  'Nombre del Comercial','OS-Proveedor',
    'costo_antes_iva', 'OS-Valor Unitario', 
]].rename(columns={'Id._x' : 'Id.'})
df_final.drop_duplicates(inplace=True)


In [33]:
df_op = df_op[[
    'Num-OP','Estado','Total Precio OP' ]]
df_op.drop_duplicates(inplace=True)

In [34]:
df_fin = df_final.merge(
    df_op,
    on="Num-OP",
    how="left"
)
df_fin.duplicated().sum()

np.int64(0)

In [35]:
# Definimos la lista de categorías oficiales para la clasificación de productos
categorias_oficiales = [
    'Accesorios', 'Bata', 'Bermuda', 'Blusa', 'Bolígrafo', 'Botas', 'Botella', 'Botón', 'Broche', 'Buzo', 'Camibuzo', 
    'Camiseta', 'Camisa', 'Cachuchera', 'Casco', 'Chaleco', 'Chaqueta', 'Cinta', 'Cofia', 'Conjunto', 'Cordón', 
    'Cremallera', 'Cuello', 'Delantal', 'Diseño', 'Embone', 'Falda', 'Gafas seguridad', 'Gorra', 'Gorro', 'Guantes', 
    'Guata', 'Hilo', 'Hoodie', 'Impermeable', 'Jean', 'Libreta', 'Lonchera', 'Mangas', 'Marquilla', 'Máscara', 'Medias', 
    'Morral', 'Ojalete', 'Overol', 'Pantalón', 'Pantaloneta', 'Pañoleta', 'Paraguas', 'Pavas', 'Polainas', 'Resorte', 
    'Ropa interior', 'Saco', 'Sastre', 'Servicios', 'Sesgo', 'Sudadera', 'Suéter', 'Tankas', 'Tapabocas', 
    'Tapaoídos', 'Tecnología', 'Termo', 'Uniformes', 'Velcro', 'Zapatos', 'Escafandra'
]

# Creamos una función para normalizar el texto, eliminando acentos y convirtiendo a minúsculas
def normalizar(texto):
    texto = str(texto).lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    return texto

# Mapeos especiales: palabras clave -> categoría oficial
MAPEOS_ESPECIALES = {
    'polo':       'Camiseta',
    'camibuso':   'Camibuzo',   # cubre "camibuso" (mal escrito) -> "Camibuzo"
    'harrington': 'Chaqueta',
    'termica':    'Chaqueta',   # cubre "térmica" también (por la normalización)
    'hoddie':     'Hoodie',     # cubre el typo "hoddie" -> "Hoodie"
    'jogger':     'Pantalón',
    't-shirt':    'Camiseta',
    'oxford':     'Camisa'
}

def clasificar(Producto):
    producto_norm = normalizar(Producto)
    
    # 1. Primero revisamos los mapeos especiales
    for keyword, categoria in MAPEOS_ESPECIALES.items():
        pattern = r'\b' + re.escape(keyword) + r'\b'
        if re.search(pattern, producto_norm):
            return categoria
    
    # 2. Luego la búsqueda genérica en categorías oficiales
    for cat in categorias_oficiales:
        cat_norm = normalizar(cat)
        pattern = r'\b' + re.escape(cat_norm) + r'\b'
        if re.search(pattern, producto_norm):
            return cat
    
    return 'Otros'

df_fin['Categoría'] = df_fin['Productos'].apply(clasificar)   

In [36]:
show(df_fin)

Loading ITables v2.7.3 from the internet... (need help?)


## Union con excel de producción

In [37]:
excell = pd.ExcelFile("C:/Users/Laura/OneDrive - MOLT/Documentos/Proveedores y clientes/SEGUIMIENTO PRODUCCION.xlsx")
df_produccion_confeccion = pd.read_excel(excell, sheet_name='Satelites', header=2)
df_produccion_bye = pd.read_excel(excell, sheet_name='Bordados-Estampados', header=2)
df_produccion_bye.columns

C:\Users\Laura\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
C:\Users\Laura\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


Index(['ID', 'ESTADO', '# OS SÁTELITE CONFECCIÓN', '#OP DETALLE', 'CLIENTE',
       'DESCRIPCION PRENDA', 'CANTIDAD O.S.',
       'FECHA ENVIO A TALLER DE SERVICIO', 'FECHA COMPROMISO SERVICIO',
       'FECHA ENTREGA BORDADO Y/O ESTAMPADO', '# DIAS EN TALLER',
       'PRECIO MARCACIÓN', 'TALLER SERVICIO', 'CANTIDAD ENTREGA SATELITE',
       'CANTIDAD PENDIENTE SATELITE', 'SEXO', 'T:U', '28', '30', '32', '34',
       '36', '38', '40', '42', '44', '46', '48', '50', '52',
       'TALLER CONFECCION', 'Unnamed: 31', 'Unnamed: 32'],
      dtype='object')

In [38]:
df_produccion_confeccion = df_produccion_confeccion[[
    '#OP DETALLE',
    '# OS SÁTELITE CONFECCIÓN',
    'CANTIDAD ENTREGA SATELITE',
    'TALLER CONFECCION'
]]

df_produccion_confeccion = df_produccion_confeccion.rename(columns={
    '#OP DETALLE' : 'OP-D', 
    '# OS SÁTELITE CONFECCIÓN' : 'Num OS',
    'TALLER CONFECCION': 'TALLER SERVICIO'})

df_produccion_confeccion['Num OS'] = df_produccion_confeccion['Num OS'].astype('Int64')

df_produccion_confeccion

,OP-D,Num OS,CANTIDAD ENTREGA SATELITE,TALLER SERVICIO
0,4968-10,12447,3.0,FLIPPER SPORT
1,5140-1,12451,55.0,DIOMEDEZ MARTINEZ
2,5140-2,12452,11.0,DIOMEDEZ MARTINEZ
3,5140-3,12454,8.0,MAUDA ALFONSO
4,5140-4,12455,33.0,MAUDA ALFONSO
...,...,...,...,...
4892,6639-22,21675,NaN,URIEL RUIZ
4893,6639-23,21683,NaN,URIEL RUIZ
4894,6650-5,21320,NaN,PATRICIA HUERTAS
4895,6625-7,21694,NaN,MARISOL BAUTISTA


In [39]:
df_produccion_bye = df_produccion_bye[[
    '#OP DETALLE',
    '# OS SÁTELITE CONFECCIÓN',
    'CANTIDAD ENTREGA SATELITE',
    'TALLER SERVICIO'
]]

df_produccion_bye = df_produccion_bye.rename(columns={'#OP DETALLE' : 'OP-D', '# OS SÁTELITE CONFECCIÓN' : 'Num OS' })

df_produccion_bye['Num OS'] = df_produccion_bye['Num OS'].astype('Int64')

df_produccion_bye = df_produccion_bye.dropna(how='all')

df_produccion = pd.concat([
    df_produccion_bye, df_produccion_confeccion
    ], ignore_index=True)

df_produccion

,OP-D,Num OS,CANTIDAD ENTREGA SATELITE,TALLER SERVICIO
0,5182-1,12642,200.0,DESIGNS AND PRINTS
1,5177-3,12647,20.0,SEIIO
2,5182-3,12650,500.0,DESIGNS AND PRINTS
3,5182-3,12651,500.0,DESIGNS AND PRINTS
4,5197-1,12652,35.0,SEIIO
...,...,...,...,...
9065,6639-22,21675,NaN,URIEL RUIZ
9066,6639-23,21683,NaN,URIEL RUIZ
9067,6650-5,21320,NaN,PATRICIA HUERTAS
9068,6625-7,21694,NaN,MARISOL BAUTISTA


In [40]:
df_produccion.drop_duplicates(inplace=True)
df_produccion[df_produccion.duplicated(keep=False)]

,OP-D,Num OS,CANTIDAD ENTREGA SATELITE,TALLER SERVICIO


In [41]:
df_union_confe = df_fin.merge(
    df_produccion,
    on='Num OS',
    how='left'
)
df_union_confe = df_union_confe.rename(columns={'OP-D_x' : 'OP-D'})


In [42]:
# Filas que NO coincidieron por 'Num OS'
df_no_coincidio = df_union_confe[df_union_confe['TALLER SERVICIO'].isna()].copy()

# Filas que SÍ coincidieron (estas ya están bien)
df_coincidencias_ok = df_union_confe[df_union_confe['TALLER SERVICIO'].notna()].copy()

# Columnas que quieres reintentar traer de df_produccion
cols_a_eliminar = ['TALLER SERVICIO', 'CANTIDAD ENTREGA SATELITE'] # Ajusta según las que necesites "limpiar"

# Eliminamos las columnas vacías para que el nuevo merge las traiga frescas
df_no_coincidio = df_no_coincidio.drop(columns=cols_a_eliminar)

# Nuevo intento de unión, ahora por 'OP-D'
df_reintento = df_no_coincidio.merge(
    df_produccion,
    on='OP-D',
    how='left'
)
dfff = pd.concat([df_coincidencias_ok, df_reintento], ignore_index=True)
dfff['Num OS'] = dfff['Num OS'].fillna(dfff['Num OS_x'])

In [43]:
# Seleccionamos solo las columnas de tipo 'object' (texto)
cols_texto = dfff.select_dtypes(include=['object']).columns

# Aplicamos mayúsculas solo a esas columnas
dfff[cols_texto] = dfff[cols_texto].apply(lambda x: x.str.upper())

In [ ]:
dfff = dfff[[
    'Num-OP', 'Secuencia', 'OP-D', 'OP Det', 'Productos', 'Cliente',
    'Fecha', 'Id.', 'Costeo_Producto', 'Costeo', 'Proceso Producción',
    'OS-Cantidad', 'Num OS', 'Nombre del Comercial', 'OS-Proveedor', 
    'costo_antes_iva', 'OS-Valor Unitario', 'Estado', 'Total Precio OP', 
    'Categoría', 'CANTIDAD ENTREGA SATELITE', 'TALLER SERVICIO',
]]

dfff['Diferencia valor unitario'] = dfff['OS-Valor Unitario'] - dfff['costo_antes_iva']
dfff['Valor total OS'] = dfff['OS-Valor Unitario'] * dfff['CANTIDAD ENTREGA SATELITE']
dfff['valor total op'] = dfff['costo_antes_iva'] * dfff['CANTIDAD ENTREGA SATELITE']
dfff['Diferencia'] = dfff['Valor total OS'] - dfff['valor total op']

In [45]:
dfff.to_excel('dffinn.xlsx', index=False)